# MIE 402 · Pre-Lab 3: Two-link pendulum
**Fall 2026 · Lab 3 · no programming required**

## What to submit on Canvas
Submit **one Word or PDF file before your own Lab 3 section starts**. Include: (1) your initial predictions and calculations, (2) the requested figures from the supplied cells, each with a caption, and (3) your own short answers to Questions 1–6. You do **not** submit code. Use your own explanations. The notebook is normally released on Canvas on the Monday before the lab; check Canvas for the authoritative posting and due time.

This pre-lab asks a different question from Spring 2026: *How do a low-angle release and two nearly identical large-angle releases compare when the same measured apparatus is used?* All simulation code is supplied. Values below are an illustrative training apparatus, **not** values to copy into the lab report; measure your actual setup in the lab.


## The device and its physical parameters
Two rigid links swing in one vertical plane. `theta1` is the upper link angle from downward vertical, and `theta2` is the **absolute** lower link angle from downward vertical. `theta2-theta1` is their relative angle. `m1,m2` are uniform bar masses; `M1,M2` are joint and lower-bob lumped masses. `L1,L2` are pivot-to-joint and joint-to-bob-center distances. Gravity acts on both bars and both lumped masses. Friction and air drag are omitted from this baseline prediction.

**Question 1 (3 points).** On a sketch, label both angles, both lengths, and four masses. If both rods hang downward, what are the two angles? Which coordinate difference gives the direction of link 2?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g=9.81
# Illustrative values only: replace with measured values during Lab 3.
p=dict(L1=0.17,L2=0.13,m1=0.010,m2=0.008,M1=0.040,M2=0.030)

def coefficients(p):
    L1,L2,m1,m2,M1,M2=[p[k] for k in ('L1','L2','m1','m2','M1','M2')]
    I11=(m1/3+M1+m2+M2)*L1**2
    I22=(m2/3+M2)*L2**2
    C=(m2/2+M2)*L1*L2
    G1=(m1/2+M1+m2+M2)*g*L1
    G2=(m2/2+M2)*g*L2
    return I11,I22,C,G1,G2

def rhs(t,y,p):
    th1,w1,th2,w2=y
    I11,I22,C,G1,G2=coefficients(p); d=th1-th2
    A=np.array([[I11,C*np.cos(d)],[C*np.cos(d),I22]])
    b=np.array([-C*np.sin(d)*w2**2-G1*np.sin(th1),
                 C*np.sin(d)*w1**2-G2*np.sin(th2)])
    a1,a2=np.linalg.solve(A,b)
    return np.array([w1,a1,w2,a2])

def simulate(deg1,deg2,duration=8,fs=200):
    t=np.arange(0,duration,1/fs); dt=1/fs
    y=np.zeros((4,len(t))); y[:,0]=np.radians([deg1,0,deg2,0])
    for i in range(len(t)-1):
        q=y[:,i]; k1=rhs(t[i],q,p)
        k2=rhs(t[i]+dt/2,q+dt*k1/2,p)
        k3=rhs(t[i]+dt/2,q+dt*k2/2,p)
        k4=rhs(t[i]+dt,q+dt*k3,p)
        y[:,i+1]=q+dt*(k1+2*k2+2*k3+k4)/6
    return t,y

def energy(y):
    I11,I22,C,G1,G2=coefficients(p)
    a,w,b,v=y
    return .5*I11*w*w+.5*I22*v*v+C*np.cos(a-b)*w*v+G1*(1-np.cos(a))+G2*(1-np.cos(b))

trials={'small':(5,5),'large A':(95,35),'large B':(96,35)}
results={name:simulate(*angles) for name,angles in trials.items()}
for name,(t,y) in results.items():
    E=energy(y); print(name, 'start angles (deg)=',trials[name],
                       'max relative energy drift=',round(np.max(np.abs(E-E[0]))/E[0],7))


**Question 2 (3 points).** Predict which case has the simplest-looking angle traces. Why do the first two releases that differ by only 1° need the *same* model parameters for a fair comparison? State one reason a real recording may show energy loss although this simulation does not.


In [ ]:
fig,axs=plt.subplots(3,2,figsize=(12,9),sharex=True)
for row,(name,(t,y)) in enumerate(results.items()):
    for col,idx in enumerate((0,2)):
        axs[row,col].plot(t,np.degrees(y[idx]),lw=1.4)
        axs[row,col].set_ylabel(name+'  angle (deg)')
        axs[row,col].set_title('upper link' if col==0 else 'lower link')
        axs[row,col].grid(alpha=.25)
for ax in axs[-1,:]: ax.set_xlabel('Time (s)')
fig.suptitle('Pre-Lab 3: two measured angles in three simulated releases')
fig.tight_layout();plt.show()


**Question 3 (4 points).** Insert this figure in your Word/PDF. Describe one visible difference between the low-angle and large-angle cases for *each* link. The large traces may be complex: do not call them “chaos” solely from appearance.


In [ ]:
ta,ya=results['large A'];tb,yb=results['large B']
sep=np.sqrt((ya[0]-yb[0])**2+(ya[2]-yb[2])**2)
fig,ax=plt.subplots(figsize=(10,4));ax.plot(ta,np.degrees(sep),color='maroon')
ax.set(xlabel='Time (s)',ylabel='combined angular separation (deg)',
       title='Difference between starts (95°,35°) and (96°,35°)')
ax.grid(alpha=.3);plt.show()
for sec in (1,2,4,6):
    k=np.argmin(abs(ta-sec));print(f'at {ta[k]:.2f} s: separation {np.degrees(sep[k]):.3f} deg')


**Question 4 (3 points).** Include the separation plot. At 1 s and 6 s, read the printed separations. Did the two starts remain 1° apart? Explain what this demonstrates and why a finite plot alone cannot establish mathematical chaos.


In [ ]:
def spectrum(t,x):
    dt=np.median(np.diff(t));f=np.fft.rfftfreq(len(t),dt)
    a=2*np.abs(np.fft.rfft((x-np.mean(x))*np.hanning(len(x))))/np.sum(np.hanning(len(x)))
    return f,a
fig,axs=plt.subplots(2,2,figsize=(11,7),sharex=True)
for row,name in enumerate(('small','large A')):
    t,y=results[name]
    for col,idx in enumerate((0,2)):
        f,a=spectrum(t,y[idx]);axs[row,col].plot(f,a,color='navy')
        axs[row,col].set_xlim(0,8);axs[row,col].set_ylabel(name+' amplitude (rad)')
        axs[row,col].set_title('upper link' if col==0 else 'lower link')
        axs[row,col].grid(alpha=.25)
for ax in axs[-1,:]:ax.set_xlabel('Frequency (Hz)')
fig.suptitle('One-sided spectra; mean removed, Hann window')
fig.tight_layout();plt.show()


**Question 5 (4 points).** Insert the spectra figure. For the low-angle case, estimate the two strongest nonzero peak frequencies (Hz) from either link. Do both links show exactly one frequency? For the large case, compare the spread of spectral content. A peak shows a frequency component in the measured angle; it is not an extra physical bob.


## Prepare for the real Lab 3
The analyzer produces `ExpData = [time_s, x1_mm, y1_mm, x2_mm, y2_mm]`. Its y-coordinate is positive upward and the positions are relative to the fixed pivot. Thus `theta1=atan2(x1,-y1)` and `theta2=atan2(x2-x1,-(y2-y1))`, followed by unwrapping. Enter the camera's **capture** frame rate, not the MP4 playback rate. Measure the real bar and bob masses and both pivot-to-center lengths.

**Question 6 (3 points).** Write a short lab plan: identify (a) the four releases, (b) data/settings to write down before recording, (c) what to check in Quick Review, and (d) files and figures to show the GTA before leaving.

**Total: 20 points.** Bring this completed Word/PDF and your predictions to your own section.
